null model을 포함해서 성능 지표를 제시해야 한다.

In [1]:
from pathlib import Path
import logging
import joblib
import numpy as np
import numpy.typing as npt
from tqdm import tqdm
from xgboost import XGBClassifier
from keras.models import load_model
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
)
import statsmodels.api as sm
from scipy.optimize import minimize_scalar, minimize
import matplotlib.pyplot as plt
from src.module.utils import load_config, get_peak_trough, process_meanbp

In [2]:
def extract_lastmbp(arr, config):
    mbp = process_meanbp(arr.reshape(-1), config)
    mbp = np.unique(mbp[np.where(mbp != 0)])
    return mbp[-1]


def get_min_size(x, p, t):
    if x[p].size < x[t].size:
        min_size = x[p].size
    else:
        min_size = x[t].size
    return min_size


def summary_stat(arr):
    x = arr.reshape(-1)
    x_mean, x_std, x_median, x_min, x_max = (
        np.mean(x),
        np.std(x),
        np.median(x),
        np.min(x),
        np.max(x),
    )
    return x_mean, x_std, x_median, x_min, x_max


def slope_intercept(t, y):
    t = np.asarray(t, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    tm, ym = t.mean(), y.mean()
    dt = t - tm
    denom = (dt * dt).sum()
    if denom == 0:
        return np.nan, np.nan
    slope = (dt * (y - ym)).sum() / denom
    return slope, ym - (slope * tm)


def extract_feature(arr, config):
    feature = []
    p, t = get_peak_trough(arr, config["detection"])
    min_size = get_min_size(arr, p, t)
    mbp = process_meanbp(arr.reshape(-1), config)
    sbp, dbp, pp = arr[p], arr[t], arr[p][:min_size] - arr[t][:min_size]
    mbp = np.unique(mbp[np.where(mbp != 0)])

    sbp_stat = summary_stat(sbp)
    dbp_stat = summary_stat(dbp)
    pp_stat = summary_stat(pp)
    mbp_stat = summary_stat(mbp)
    trend_per_beat, _ = slope_intercept(np.arange(mbp.size), mbp)
    trend_per_sec = trend_per_beat * (mbp.size / 30.0)
    half = mbp.size // 2
    slope_per_beat, _ = slope_intercept(np.arange(mbp.size)[half:], mbp[half:])
    slope_per_sec = slope_per_beat * (mbp.size / 30.0)
    last_mbp = mbp[-1]

    feature.extend(
        [
            *sbp_stat,
            *dbp_stat,
            *pp_stat,
            *mbp_stat,
            trend_per_sec,
            slope_per_sec,
            last_mbp,
        ]
    )
    return feature


def brier_skill_score(y_true, y_prob):
    y_true = np.asarray(y_true, dtype=float)
    bs_model = brier_score_loss(y_true, y_prob)
    p = y_true.mean()  # 평가셋 자연 유병률
    bs_ref = p * (1.0 - p)  # 항상 p를 예측하는 무정보 모델
    return 1.0 - bs_model / bs_ref, bs_model, bs_ref


# def calib_metrics(y, p):
#     eps = 1e-12
#     lp = np.log(np.clip(p, eps, 1 - eps) / np.clip(1 - p, eps, 1 - eps))
#     slope = sm.Logit(y, sm.add_constant(lp)).fit(disp=0).params[1]
#     intercept = (
#         sm.Logit(y, sm.add_constant(lp, has_constant="add"), offset=lp)
#         .fit(disp=0)
#         .params[0]
#     )
#     oe = y.sum() / p.sum()
#     return intercept, slope, oe


def calib_metrics(y, p):
    y = np.asarray(y, float)
    p = np.clip(np.asarray(p, float), 1e-6, 1 - 1e-6)
    lp = np.log(p / (1 - p))
    oe = y.sum() / p.sum()

    # slope: a + b*lp 동시 추정
    f = lambda t: -np.sum(y * (t[0] + t[1] * lp) - np.logaddexp(0, t[0] + t[1] * lp))
    slope = minimize(f, x0=[0.0, 1.0], method="L-BFGS-B").x[1]

    # intercept: slope를 1로 고정(offset)
    g = lambda a: -np.sum(y * (a + lp) - np.logaddexp(0, a + lp))
    icpt = minimize_scalar(g, bounds=(-10, 10), method="bounded").x

    return icpt, slope, oe


def scale_input(input) -> npt.NDArray:
    return input.astype(np.float32) / 200

In [3]:
def patient_level_bootstrap(cid, seed, n, ref_y, test_y, prob_y, thrs: list):
    # Patient-level bootstrap resampling
    pat_ids = np.unique(cid)
    idx_by_pat = {p: np.where(cid == p)[0] for p in pat_ids}
    np.random.seed(seed)  # 42

    auroc_list, delta_list, auprc_list = [], [], []
    bs_list, bss_list = [], []
    icpt_list, slope_list, oe_list = [], [], []
    acc_80, sen_80, spe_80, ppv_80, npv_80 = [], [], [], [], []
    acc_90, sen_90, spe_90, ppv_90, npv_90 = [], [], [], [], []
    acc_95, sen_95, spe_95, ppv_95, npv_95 = [], [], [], [], []
    for i in tqdm(range(n), ncols=75):
        sampled = np.random.choice(pat_ids, size=len(pat_ids), replace=True)
        idx = np.concatenate([idx_by_pat[p] for p in sampled])
        sampled_ref_test = ref_y[idx]
        sampled_test_y, sampled_prob_y = test_y[idx], prob_y[idx]

        base_auroc = roc_auc_score(sampled_test_y, sampled_ref_test)
        auroc = roc_auc_score(sampled_test_y, sampled_prob_y)
        auprc = average_precision_score(sampled_test_y, sampled_prob_y)
        bss, bs, bs_null = brier_skill_score(sampled_test_y, sampled_prob_y)
        intercept, slope, oe = calib_metrics(sampled_test_y, sampled_prob_y)

        auroc_list.append(auroc)
        delta_list.append(auroc - base_auroc)
        auprc_list.append(auprc)
        bs_list.append(bs)
        bss_list.append(bss)
        icpt_list.append(intercept)
        slope_list.append(slope)
        oe_list.append(oe)

        pred = (sampled_prob_y >= thrs[0]).astype(int)  # Spe (train): 80%
        tn, fp, fn, tp = confusion_matrix(sampled_test_y, pred).ravel()
        acc_80.append((tp + tn) / (tp + tn + fp + fn))
        sen_80.append(tp / (tp + fn))
        spe_80.append(tn / (tn + fp))
        ppv_80.append(tp / (tp + fp) if (tp + fp) else np.nan)
        npv_80.append(tn / (tn + fn) if (tn + fn) else np.nan)

        pred = (sampled_prob_y >= thrs[1]).astype(int)  # Spe (train): 90%
        tn, fp, fn, tp = confusion_matrix(sampled_test_y, pred).ravel()
        acc_90.append((tp + tn) / (tp + tn + fp + fn))
        sen_90.append(tp / (tp + fn))
        spe_90.append(tn / (tn + fp))
        ppv_90.append(tp / (tp + fp) if (tp + fp) else np.nan)
        npv_90.append(tn / (tn + fn) if (tn + fn) else np.nan)

        pred = (sampled_prob_y >= thrs[2]).astype(int)  # Spe (train): 95%
        tn, fp, fn, tp = confusion_matrix(sampled_test_y, pred).ravel()
        acc_95.append((tp + tn) / (tp + tn + fp + fn))
        sen_95.append(tp / (tp + fn))
        spe_95.append(tn / (tn + fp))
        ppv_95.append(tp / (tp + fp) if (tp + fp) else np.nan)
        npv_95.append(tn / (tn + fn) if (tn + fn) else np.nan)

    print("95% CI (without gray)")
    print(
        f"AUROC: ({np.percentile(auroc_list, 2.5):.3f}-{np.percentile(auroc_list, 97.5):.3f})"
    )
    print(
        f"∆AUROC (vs. Ref): ({np.percentile(delta_list, 2.5):.3f}-{np.percentile(delta_list, 97.5):.3f})"
    )
    print(
        f"AUPRC: ({np.percentile(auprc_list, 2.5):.3f}-{np.percentile(auprc_list, 97.5):.3f})"
    )
    print(
        f"Brier score: ({np.percentile(bs_list, 2.5):.3f}-{np.percentile(bs_list, 97.5):.3f})"
    )
    print(
        f"Brier skill score: ({np.percentile(bss_list, 2.5):.3f}-{np.percentile(bss_list, 97.5):.3f})"
    )
    print(
        f"Intercept(calib): ({np.percentile(icpt_list, 2.5):.3f}-{np.percentile(icpt_list, 97.5):.3f})"
    )
    print(
        f"Slope(calib): ({np.percentile(slope_list, 2.5):.3f}-{np.percentile(slope_list, 97.5):.3f})"
    )
    print(
        f"OE ratio: ({np.percentile(oe_list, 2.5):.3f}-{np.percentile(oe_list, 97.5):.3f})"
    )
    print("Spe80% (train):")
    print(
        f"- Accuracy: ({np.percentile(acc_80, 2.5):.3f}-{np.percentile(acc_80, 97.5):.3f})"
    )
    print(
        f"- Sen: ({np.percentile(sen_80, 2.5):.3f}-{np.percentile(sen_80, 97.5):.3f})"
    )
    print(
        f"- Spe: ({np.percentile(spe_80, 2.5):.3f}-{np.percentile(spe_80, 97.5):.3f})"
    )
    print(
        f"- PPV: ({np.percentile(ppv_80, 2.5):.3f}-{np.percentile(ppv_80, 97.5):.3f})"
    )
    print(
        f"- NPV: ({np.percentile(npv_80, 2.5):.3f}-{np.percentile(npv_80, 97.5):.3f})"
    )
    print("Spe90% (train):")
    print(
        f"- Accuracy: ({np.percentile(acc_90, 2.5):.3f}-{np.percentile(acc_90, 97.5):.3f})"
    )
    print(
        f"- Sen: ({np.percentile(sen_90, 2.5):.3f}-{np.percentile(sen_90, 97.5):.3f})"
    )
    print(
        f"- Spe: ({np.percentile(spe_90, 2.5):.3f}-{np.percentile(spe_90, 97.5):.3f})"
    )
    print(
        f"- PPV: ({np.percentile(ppv_90, 2.5):.3f}-{np.percentile(ppv_90, 97.5):.3f})"
    )
    print(
        f"- NPV: ({np.percentile(npv_90, 2.5):.3f}-{np.percentile(npv_90, 97.5):.3f})"
    )
    print("Spe95% (train):")
    print(
        f"- Accuracy: ({np.percentile(acc_95, 2.5):.3f}-{np.percentile(acc_95, 97.5):.3f})"
    )
    print(
        f"- Sen: ({np.percentile(sen_95, 2.5):.3f}-{np.percentile(sen_95, 97.5):.3f})"
    )
    print(
        f"- Spe: ({np.percentile(spe_95, 2.5):.3f}-{np.percentile(spe_95, 97.5):.3f})"
    )
    print(
        f"- PPV: ({np.percentile(ppv_95, 2.5):.3f}-{np.percentile(ppv_95, 97.5):.3f})"
    )
    print(
        f"- NPV: ({np.percentile(npv_95, 2.5):.3f}-{np.percentile(npv_95, 97.5):.3f})"
    )

In [4]:
config = load_config("../../config/conf.yaml")
src_path = Path(config["data_path"]).expanduser() / "mover" / "06.train_val_test_new"
model_path = Path(config["model_path"]).expanduser() / "revision_seed42"
strategy = "hypophetversion2"

In [5]:
test_cid = np.load(src_path / f"mover_{strategy}_test_cid.npy")
test_X = np.load(src_path / f"mover_{strategy}_test_X.npy")
test_y = np.load(src_path / f"mover_{strategy}_test_y.npy")
print(test_cid.shape, test_X.shape, test_y.shape)

(25793,) (25793, 3000) (25793,)


In [6]:
test_X_feat = np.array([extract_feature(x, config) for x in tqdm(test_X, ncols=75)])
print(test_X_feat.shape)

100%|██████████████████████████████| 25793/25793 [00:08<00:00, 2931.31it/s]

(25793, 23)


In [7]:
test_X_s = scale_input(test_X)

# Null model (only BS needed)

In [9]:
prob_y = np.repeat(np.mean(test_y), test_y.size)
null_model_bs = brier_score_loss(test_y, prob_y)
print(f"Prevalence: {np.mean(test_y)}")
print(f"BS(null model): {null_model_bs}")

Prevalence: 0.36036909238940795
BS(null model): 0.23050320963984225


# MBP Baseline

In [10]:
# eval 목적
lastmbp_test = np.array([extract_lastmbp(x, config) for x in tqdm(test_X, ncols=75)])
print(lastmbp_test.shape)

100%|██████████████████████████████| 25793/25793 [00:05<00:00, 5025.33it/s]

(25793,)


In [11]:
print(f"AUROC: {roc_auc_score(test_y, -lastmbp_test)}")
print(f"PRAUC: {average_precision_score(test_y, -lastmbp_test)}")

AUROC: 0.6975586360542114
PRAUC: 0.590331425588746


In [12]:
print("Specificity(train): 0.8")
pred = (lastmbp_test < 80.36).astype(int)  # Spe (train): 80%
tn, fp, fn, tp = confusion_matrix(test_y, pred).ravel()
acc = (tp + tn) / (tp + tn + fp + fn)
sen = tp / (tp + fn)
spe = tn / (tn + fp)
ppv = tp / (tp + fp) if (tp + fp) else np.nan
npv = tn / (tn + fn) if (tn + fn) else np.nan
print(f"Acc: {acc:.3f}")
print(f"Sen: {sen:.3f}")
print(f"Spe: {spe:.3f}")
print(f"PPV: {ppv:.3f}")
print(f"NPV: {npv:.3f}")
print(classification_report(test_y, np.where(lastmbp_test < 80.36, 1, 0)))

Specificity(train): 0.8
Acc: 0.686
Sen: 0.528
Spe: 0.775
PPV: 0.570
NPV: 0.745
              precision    recall  f1-score   support

         0.0       0.74      0.78      0.76     16498
         1.0       0.57      0.53      0.55      9295

    accuracy                           0.69     25793
   macro avg       0.66      0.65      0.65     25793
weighted avg       0.68      0.69      0.68     25793



In [13]:
print("Specificity(train): 0.9")
pred = (lastmbp_test < 76.38).astype(int)  # Spe (train): 80%
tn, fp, fn, tp = confusion_matrix(test_y, pred).ravel()
acc = (tp + tn) / (tp + tn + fp + fn)
sen = tp / (tp + fn)
spe = tn / (tn + fp)
ppv = tp / (tp + fp) if (tp + fp) else np.nan
npv = tn / (tn + fn) if (tn + fn) else np.nan
print(f"Acc: {acc:.3f}")
print(f"Sen: {sen:.3f}")
print(f"Spe: {spe:.3f}")
print(f"PPV: {ppv:.3f}")
print(f"NPV: {npv:.3f}")
print(classification_report(test_y, np.where(lastmbp_test < 76.38, 1, 0)))

Specificity(train): 0.9
Acc: 0.698
Sen: 0.361
Spe: 0.887
PPV: 0.644
NPV: 0.711
              precision    recall  f1-score   support

         0.0       0.71      0.89      0.79     16498
         1.0       0.64      0.36      0.46      9295

    accuracy                           0.70     25793
   macro avg       0.68      0.62      0.63     25793
weighted avg       0.69      0.70      0.67     25793



In [14]:
print("Specificity(train): 0.95")
pred = (lastmbp_test < 73.68).astype(int)  # Spe (train): 80%
tn, fp, fn, tp = confusion_matrix(test_y, pred).ravel()
acc = (tp + tn) / (tp + tn + fp + fn)
sen = tp / (tp + fn)
spe = tn / (tn + fp)
ppv = tp / (tp + fp) if (tp + fp) else np.nan
npv = tn / (tn + fn) if (tn + fn) else np.nan
print(f"Acc: {acc:.3f}")
print(f"Sen: {sen:.3f}")
print(f"Spe: {spe:.3f}")
print(f"PPV: {ppv:.3f}")
print(f"NPV: {npv:.3f}")
print(classification_report(test_y, np.where(lastmbp_test < 73.68, 1, 0)))

Specificity(train): 0.95
Acc: 0.691
Sen: 0.239
Spe: 0.946
PPV: 0.713
NPV: 0.688
              precision    recall  f1-score   support

         0.0       0.69      0.95      0.80     16498
         1.0       0.71      0.24      0.36      9295

    accuracy                           0.69     25793
   macro avg       0.70      0.59      0.58     25793
weighted avg       0.70      0.69      0.64     25793



In [15]:
# Patient-level bootstrap resampling
pat_ids = np.unique(test_cid)
idx_by_pat = {p: np.where(test_cid == p)[0] for p in pat_ids}
np.random.seed(config["seed"])  # 42

n = 2000
auroc_list, auprc_list = [], []
acc_80, sen_80, spe_80, ppv_80, npv_80 = [], [], [], [], []
acc_90, sen_90, spe_90, ppv_90, npv_90 = [], [], [], [], []
acc_95, sen_95, spe_95, ppv_95, npv_95 = [], [], [], [], []
for i in tqdm(range(n), ncols=75):
    sampled = np.random.choice(pat_ids, size=len(pat_ids), replace=True)
    idx = np.concatenate([idx_by_pat[p] for p in sampled])
    sampled_mbp_test, sampled_test_y = lastmbp_test[idx], test_y[idx]
    auroc_list.append(roc_auc_score(sampled_test_y, -sampled_mbp_test))
    auprc_list.append(average_precision_score(sampled_test_y, -sampled_mbp_test))

    pred = (sampled_mbp_test < 80.36).astype(int)  # Spe (train): 80%
    tn, fp, fn, tp = confusion_matrix(sampled_test_y, pred).ravel()
    acc_80.append((tp + tn) / (tp + tn + fp + fn))
    sen_80.append(tp / (tp + fn))
    spe_80.append(tn / (tn + fp))
    ppv_80.append(tp / (tp + fp) if (tp + fp) else np.nan)
    npv_80.append(tn / (tn + fn) if (tn + fn) else np.nan)

    pred = (sampled_mbp_test < 76.38).astype(int)  # Spe (train): 90%
    tn, fp, fn, tp = confusion_matrix(sampled_test_y, pred).ravel()
    acc_90.append((tp + tn) / (tp + tn + fp + fn))
    sen_90.append(tp / (tp + fn))
    spe_90.append(tn / (tn + fp))
    ppv_90.append(tp / (tp + fp) if (tp + fp) else np.nan)
    npv_90.append(tn / (tn + fn) if (tn + fn) else np.nan)

    pred = (sampled_mbp_test < 73.68).astype(int)  # Spe (train): 95%
    tn, fp, fn, tp = confusion_matrix(sampled_test_y, pred).ravel()
    acc_95.append((tp + tn) / (tp + tn + fp + fn))
    sen_95.append(tp / (tp + fn))
    spe_95.append(tn / (tn + fp))
    ppv_95.append(tp / (tp + fp) if (tp + fp) else np.nan)
    npv_95.append(tn / (tn + fn) if (tn + fn) else np.nan)

print("95% CI (without gray)")
print(
    f"AUROC: ({np.percentile(auroc_list, 2.5):.3f}-{np.percentile(auroc_list, 97.5):.3f})"
)
print(
    f"AUPRC: ({np.percentile(auprc_list, 2.5):.3f}-{np.percentile(auprc_list, 97.5):.3f})"
)
print("Spe80% (train):")
print(
    f"- Accuracy: ({np.percentile(acc_80, 2.5):.3f}-{np.percentile(acc_80, 97.5):.3f})"
)
print(f"- Sen: ({np.percentile(sen_80, 2.5):.3f}-{np.percentile(sen_80, 97.5):.3f})")
print(f"- Spe: ({np.percentile(spe_80, 2.5):.3f}-{np.percentile(spe_80, 97.5):.3f})")
print(f"- PPV: ({np.percentile(ppv_80, 2.5):.3f}-{np.percentile(ppv_80, 97.5):.3f})")
print(f"- NPV: ({np.percentile(npv_80, 2.5):.3f}-{np.percentile(npv_80, 97.5):.3f})")
print("Spe90% (train):")
print(
    f"- Accuracy: ({np.percentile(acc_90, 2.5):.3f}-{np.percentile(acc_90, 97.5):.3f})"
)
print(f"- Sen: ({np.percentile(sen_90, 2.5):.3f}-{np.percentile(sen_90, 97.5):.3f})")
print(f"- Spe: ({np.percentile(spe_90, 2.5):.3f}-{np.percentile(spe_90, 97.5):.3f})")
print(f"- PPV: ({np.percentile(ppv_90, 2.5):.3f}-{np.percentile(ppv_90, 97.5):.3f})")
print(f"- NPV: ({np.percentile(npv_90, 2.5):.3f}-{np.percentile(npv_90, 97.5):.3f})")
print("Spe95% (train):")
print(
    f"- Accuracy: ({np.percentile(acc_95, 2.5):.3f}-{np.percentile(acc_95, 97.5):.3f})"
)
print(f"- Sen: ({np.percentile(sen_95, 2.5):.3f}-{np.percentile(sen_95, 97.5):.3f})")
print(f"- Spe: ({np.percentile(spe_95, 2.5):.3f}-{np.percentile(spe_95, 97.5):.3f})")
print(f"- PPV: ({np.percentile(ppv_95, 2.5):.3f}-{np.percentile(ppv_95, 97.5):.3f})")
print(f"- NPV: ({np.percentile(npv_95, 2.5):.3f}-{np.percentile(npv_95, 97.5):.3f})")

100%|██████████████████████████████████| 2000/2000 [00:31<00:00, 62.75it/s]

95% CI (without gray)
AUROC: (0.674-0.721)
AUPRC: (0.557-0.623)
Spe80% (train):
- Accuracy: (0.667-0.707)
- Sen: (0.495-0.560)
- Spe: (0.753-0.797)
- PPV: (0.534-0.605)
- NPV: (0.714-0.774)
Spe90% (train):
- Accuracy: (0.676-0.719)
- Sen: (0.333-0.391)
- Spe: (0.873-0.901)
- PPV: (0.606-0.680)
- NPV: (0.680-0.740)
Spe95% (train):
- Accuracy: (0.666-0.715)
- Sen: (0.215-0.265)
- Spe: (0.937-0.954)
- PPV: (0.672-0.750)
- NPV: (0.656-0.717)


# Log Reg

In [18]:
lrpipe = joblib.load(model_path / "lrpipe_baseline.joblib")

In [19]:
prob_y = lrpipe["model"].predict_proba(test_X_feat)[:, 1]

In [20]:
auroc = roc_auc_score(test_y, prob_y)
auprc = average_precision_score(test_y, prob_y)
bss, bs, bs_null = brier_skill_score(test_y, prob_y)
intercept, slope, oe = calib_metrics(test_y, prob_y)
print(f"AUROC: {auroc:.3f}")
print(f"AUPRC: {auprc:.3f}")
print(f"Brier={bs:.3f}, Null Brier={bs_null:.3f}, BSS={bss:.3f}")
print(f"Intercept={intercept:.3f}, Slope={slope:.3f}, OE ratio={oe:.3f}")

AUROC: 0.740
AUPRC: 0.594
Brier=0.227, Null Brier=0.231, BSS=0.015
Intercept=-0.873, Slope=0.814, OE ratio=0.681


In [21]:
print("Specificity(train): 0.8")
pred = (prob_y >= 0.61).astype(int)  # Spe (train): 80%
tn, fp, fn, tp = confusion_matrix(test_y, pred).ravel()
acc = (tp + tn) / (tp + tn + fp + fn)
sen = tp / (tp + fn)
spe = tn / (tn + fp)
ppv = tp / (tp + fp) if (tp + fp) else np.nan
npv = tn / (tn + fn) if (tn + fn) else np.nan
print(f"Acc: {acc:.3f}")
print(f"Sen: {sen:.3f}")
print(f"Spe: {spe:.3f}")
print(f"PPV: {ppv:.3f}")
print(f"NPV: {npv:.3f}")
print(classification_report(test_y, np.where(prob_y >= 0.61, 1, 0)))

Specificity(train): 0.8
Acc: 0.690
Sen: 0.670
Spe: 0.701
PPV: 0.558
NPV: 0.790
              precision    recall  f1-score   support

         0.0       0.79      0.70      0.74     16498
         1.0       0.56      0.67      0.61      9295

    accuracy                           0.69     25793
   macro avg       0.67      0.69      0.68     25793
weighted avg       0.71      0.69      0.69     25793



In [22]:
print("Specificity(train): 0.9")
pred = (prob_y >= 0.69).astype(int)  # Spe (train): 80%
tn, fp, fn, tp = confusion_matrix(test_y, pred).ravel()
acc = (tp + tn) / (tp + tn + fp + fn)
sen = tp / (tp + fn)
spe = tn / (tn + fp)
ppv = tp / (tp + fp) if (tp + fp) else np.nan
npv = tn / (tn + fn) if (tn + fn) else np.nan
print(f"Acc: {acc:.3f}")
print(f"Sen: {sen:.3f}")
print(f"Spe: {spe:.3f}")
print(f"PPV: {ppv:.3f}")
print(f"NPV: {npv:.3f}")
print(classification_report(test_y, np.where(prob_y >= 0.69, 1, 0)))

Specificity(train): 0.9
Acc: 0.707
Sen: 0.521
Spe: 0.811
PPV: 0.609
NPV: 0.750
              precision    recall  f1-score   support

         0.0       0.75      0.81      0.78     16498
         1.0       0.61      0.52      0.56      9295

    accuracy                           0.71     25793
   macro avg       0.68      0.67      0.67     25793
weighted avg       0.70      0.71      0.70     25793



In [23]:
print("Specificity(train): 0.95")
pred = (prob_y >= 0.75).astype(int)  # Spe (train): 80%
tn, fp, fn, tp = confusion_matrix(test_y, pred).ravel()
acc = (tp + tn) / (tp + tn + fp + fn)
sen = tp / (tp + fn)
spe = tn / (tn + fp)
ppv = tp / (tp + fp) if (tp + fp) else np.nan
npv = tn / (tn + fn) if (tn + fn) else np.nan
print(f"Acc: {acc:.3f}")
print(f"Sen: {sen:.3f}")
print(f"Spe: {spe:.3f}")
print(f"PPV: {ppv:.3f}")
print(f"NPV: {npv:.3f}")
print(classification_report(test_y, np.where(prob_y >= 0.75, 1, 0)))

Specificity(train): 0.95
Acc: 0.708
Sen: 0.384
Spe: 0.890
PPV: 0.664
NPV: 0.719
              precision    recall  f1-score   support

         0.0       0.72      0.89      0.80     16498
         1.0       0.66      0.38      0.49      9295

    accuracy                           0.71     25793
   macro avg       0.69      0.64      0.64     25793
weighted avg       0.70      0.71      0.68     25793



In [24]:
patient_level_bootstrap(
    cid=test_cid,
    seed=config["seed"],
    n=2000,
    ref_y=-lastmbp_test,
    test_y=test_y,
    prob_y=prob_y,
    thrs=[0.61, 0.69, 0.75],
)

100%|██████████████████████████████████| 2000/2000 [00:52<00:00, 38.36it/s]

95% CI (without gray)
AUROC: (0.720-0.761)
∆AUROC (vs. Ref): (0.026-0.059)
AUPRC: (0.559-0.630)
Brier score: (0.218-0.236)
Brier skill score: (-0.042-0.066)
Intercept(calib): (-1.008--0.743)
Slope(calib): (0.719-0.920)
OE ratio: (0.633-0.730)
Spe80% (train):
- Accuracy: (0.672-0.708)
- Sen: (0.641-0.700)
- Spe: (0.674-0.727)
- PPV: (0.525-0.591)
- NPV: (0.761-0.817)
Spe90% (train):
- Accuracy: (0.687-0.726)
- Sen: (0.490-0.553)
- Spe: (0.789-0.832)
- PPV: (0.574-0.642)
- NPV: (0.720-0.778)
Spe95% (train):
- Accuracy: (0.685-0.730)
- Sen: (0.352-0.416)
- Spe: (0.873-0.905)
- PPV: (0.627-0.701)
- NPV: (0.689-0.748)


# XGB

In [24]:
loaded_xgb = XGBClassifier()
loaded_xgb.load_model(model_path / "xgb_baseline.json")

In [25]:
prob_y = loaded_xgb.predict_proba(test_X_feat)[:, 1]

In [26]:
auroc = roc_auc_score(test_y, prob_y)
auprc = average_precision_score(test_y, prob_y)
bss, bs, bs_null = brier_skill_score(test_y, prob_y)
intercept, slope, oe = calib_metrics(test_y, prob_y)
print(f"AUROC: {auroc:.3f}")
print(f"AUPRC: {auprc:.3f}")
print(f"Brier={bs:.3f}, Null Brier={bs_null:.3f}, BSS={bss:.3f}")
print(f"Intercept={intercept:.3f}, Slope={slope:.3f}, OE ratio={oe:.3f}")

AUROC: 0.741
AUPRC: 0.616
Brier=0.223, Null Brier=0.231, BSS=0.031
Intercept=-0.906, Slope=0.688, OE ratio=0.692


In [29]:
print("Specificity(train): 0.8")
pred = (prob_y >= 0.59).astype(int)  # Spe (train): 80%
tn, fp, fn, tp = confusion_matrix(test_y, pred).ravel()
acc = (tp + tn) / (tp + tn + fp + fn)
sen = tp / (tp + fn)
spe = tn / (tn + fp)
ppv = tp / (tp + fp) if (tp + fp) else np.nan
npv = tn / (tn + fn) if (tn + fn) else np.nan
print(f"Acc: {acc:.3f}")
print(f"Sen: {sen:.3f}")
print(f"Spe: {spe:.3f}")
print(f"PPV: {ppv:.3f}")
print(f"NPV: {npv:.3f}")
print(classification_report(test_y, np.where(prob_y >= 0.59, 1, 0)))

Specificity(train): 0.8
Acc: 0.683
Sen: 0.634
Spe: 0.711
PPV: 0.553
NPV: 0.775
              precision    recall  f1-score   support

         0.0       0.78      0.71      0.74     16498
         1.0       0.55      0.63      0.59      9295

    accuracy                           0.68     25793
   macro avg       0.66      0.67      0.67     25793
weighted avg       0.70      0.68      0.69     25793



In [30]:
print("Specificity(train): 0.9")
pred = (prob_y >= 0.72).astype(int)  # Spe (train): 80%
tn, fp, fn, tp = confusion_matrix(test_y, pred).ravel()
acc = (tp + tn) / (tp + tn + fp + fn)
sen = tp / (tp + fn)
spe = tn / (tn + fp)
ppv = tp / (tp + fp) if (tp + fp) else np.nan
npv = tn / (tn + fn) if (tn + fn) else np.nan
print(f"Acc: {acc:.3f}")
print(f"Sen: {sen:.3f}")
print(f"Spe: {spe:.3f}")
print(f"PPV: {ppv:.3f}")
print(f"NPV: {npv:.3f}")
print(classification_report(test_y, np.where(prob_y >= 0.72, 1, 0)))

Specificity(train): 0.9
Acc: 0.708
Sen: 0.472
Spe: 0.841
PPV: 0.626
NPV: 0.739
              precision    recall  f1-score   support

         0.0       0.74      0.84      0.79     16498
         1.0       0.63      0.47      0.54      9295

    accuracy                           0.71     25793
   macro avg       0.68      0.66      0.66     25793
weighted avg       0.70      0.71      0.70     25793



In [31]:
print("Specificity(train): 0.95")
pred = (prob_y >= 0.82).astype(int)  # Spe (train): 80%
tn, fp, fn, tp = confusion_matrix(test_y, pred).ravel()
acc = (tp + tn) / (tp + tn + fp + fn)
sen = tp / (tp + fn)
spe = tn / (tn + fp)
ppv = tp / (tp + fp) if (tp + fp) else np.nan
npv = tn / (tn + fn) if (tn + fn) else np.nan
print(f"Acc: {acc:.3f}")
print(f"Sen: {sen:.3f}")
print(f"Spe: {spe:.3f}")
print(f"PPV: {ppv:.3f}")
print(f"NPV: {npv:.3f}")
print(classification_report(test_y, np.where(prob_y >= 0.82, 1, 0)))

Specificity(train): 0.95
Acc: 0.707
Sen: 0.328
Spe: 0.921
PPV: 0.701
NPV: 0.709
              precision    recall  f1-score   support

         0.0       0.71      0.92      0.80     16498
         1.0       0.70      0.33      0.45      9295

    accuracy                           0.71     25793
   macro avg       0.70      0.62      0.62     25793
weighted avg       0.71      0.71      0.67     25793



In [32]:
patient_level_bootstrap(
    cid=test_cid,
    seed=config["seed"],
    n=2000,
    ref_y=-lastmbp_test,
    test_y=test_y,
    prob_y=prob_y,
    thrs=[0.59, 0.72, 0.82],
)

100%|██████████████████████████████████| 2000/2000 [01:43<00:00, 19.38it/s]

95% CI (without gray)
AUROC: (0.721-0.761)
∆AUROC (vs. Ref): (0.027-0.059)
AUPRC: (0.585-0.648)
Brier score: (0.214-0.232)
Brier skill score: (-0.023-0.079)
Intercept(calib): (-1.055--0.764)
Slope(calib): (0.626-0.757)
OE ratio: (0.643-0.740)
Spe80% (train):
- Accuracy: (0.666-0.701)
- Sen: (0.606-0.663)
- Spe: (0.687-0.735)
- PPV: (0.519-0.584)
- NPV: (0.745-0.802)
Spe90% (train):
- Accuracy: (0.688-0.728)
- Sen: (0.442-0.503)
- Spe: (0.824-0.857)
- PPV: (0.592-0.658)
- NPV: (0.709-0.766)
Spe95% (train):
- Accuracy: (0.685-0.730)
- Sen: (0.301-0.356)
- Spe: (0.910-0.931)
- PPV: (0.667-0.734)
- NPV: (0.679-0.737)


# CNN

In [29]:
loaded_model = load_model(model_path / "cnn.model.keras")

2026-10-08 03:11:04.689910: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M5 Max
2026-10-08 03:11:04.689963: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 128.00 GB
2026-10-08 03:11:04.689977: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 53.76 GB
2026-10-08 03:11:04.690010: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-10-08 03:11:04.690026: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [30]:
prob_y = loaded_model.predict(test_X_s).reshape(-1)

 64/807 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step

2026-10-08 03:11:05.060059: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


807/807 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [31]:
auroc = roc_auc_score(test_y, prob_y)
auprc = average_precision_score(test_y, prob_y)
bss, bs, bs_null = brier_skill_score(test_y, prob_y)
intercept, slope, oe = calib_metrics(test_y, prob_y)
print(f"AUROC: {auroc:.3f}")
print(f"AUPRC: {auprc:.3f}")
print(f"Brier={bs:.3f}, Null Brier={bs_null:.3f}, BSS={bss:.3f}")
print(f"Intercept={intercept:.3f}, Slope={slope:.3f}, OE ratio={oe:.3f}")

AUROC: 0.732
AUPRC: 0.609
Brier=0.214, Null Brier=0.231, BSS=0.070
Intercept=-0.715, Slope=0.768, OE ratio=0.730


In [32]:
print("Specificity(train): 0.8")
pred = (prob_y >= 0.58).astype(int)  # Spe (train): 80%
tn, fp, fn, tp = confusion_matrix(test_y, pred).ravel()
acc = (tp + tn) / (tp + tn + fp + fn)
sen = tp / (tp + fn)
spe = tn / (tn + fp)
ppv = tp / (tp + fp) if (tp + fp) else np.nan
npv = tn / (tn + fn) if (tn + fn) else np.nan
print(f"Acc: {acc:.3f}")
print(f"Sen: {sen:.3f}")
print(f"Spe: {spe:.3f}")
print(f"PPV: {ppv:.3f}")
print(f"NPV: {npv:.3f}")
print(classification_report(test_y, np.where(prob_y >= 0.58, 1, 0)))

Specificity(train): 0.8
Acc: 0.699
Sen: 0.589
Spe: 0.761
PPV: 0.581
NPV: 0.767
              precision    recall  f1-score   support

         0.0       0.77      0.76      0.76     16498
         1.0       0.58      0.59      0.59      9295

    accuracy                           0.70     25793
   macro avg       0.67      0.67      0.67     25793
weighted avg       0.70      0.70      0.70     25793



In [33]:
print("Specificity(train): 0.9")
pred = (prob_y >= 0.71).astype(int)  # Spe (train): 80%
tn, fp, fn, tp = confusion_matrix(test_y, pred).ravel()
acc = (tp + tn) / (tp + tn + fp + fn)
sen = tp / (tp + fn)
spe = tn / (tn + fp)
ppv = tp / (tp + fp) if (tp + fp) else np.nan
npv = tn / (tn + fn) if (tn + fn) else np.nan
print(f"Acc: {acc:.3f}")
print(f"Sen: {sen:.3f}")
print(f"Spe: {spe:.3f}")
print(f"PPV: {ppv:.3f}")
print(f"NPV: {npv:.3f}")
print(classification_report(test_y, np.where(prob_y >= 0.71, 1, 0)))

Specificity(train): 0.9
Acc: 0.711
Sen: 0.427
Spe: 0.870
PPV: 0.650
NPV: 0.729
              precision    recall  f1-score   support

         0.0       0.73      0.87      0.79     16498
         1.0       0.65      0.43      0.52      9295

    accuracy                           0.71     25793
   macro avg       0.69      0.65      0.65     25793
weighted avg       0.70      0.71      0.69     25793



In [34]:
print("Specificity(train): 0.95")
pred = (prob_y >= 0.79).astype(int)  # Spe (train): 80%
tn, fp, fn, tp = confusion_matrix(test_y, pred).ravel()
acc = (tp + tn) / (tp + tn + fp + fn)
sen = tp / (tp + fn)
spe = tn / (tn + fp)
ppv = tp / (tp + fp) if (tp + fp) else np.nan
npv = tn / (tn + fn) if (tn + fn) else np.nan
print(f"Acc: {acc:.3f}")
print(f"Sen: {sen:.3f}")
print(f"Spe: {spe:.3f}")
print(f"PPV: {ppv:.3f}")
print(f"NPV: {npv:.3f}")
print(classification_report(test_y, np.where(prob_y >= 0.79, 1, 0)))

Specificity(train): 0.95
Acc: 0.706
Sen: 0.314
Spe: 0.927
PPV: 0.707
NPV: 0.706
              precision    recall  f1-score   support

         0.0       0.71      0.93      0.80     16498
         1.0       0.71      0.31      0.43      9295

    accuracy                           0.71     25793
   macro avg       0.71      0.62      0.62     25793
weighted avg       0.71      0.71      0.67     25793



In [35]:
patient_level_bootstrap(
    cid=test_cid,
    seed=config["seed"],
    n=2000,
    ref_y=-lastmbp_test,
    test_y=test_y,
    prob_y=prob_y,
    thrs=[0.58, 0.71, 0.79],
)

100%|██████████████████████████████████| 2000/2000 [01:34<00:00, 21.19it/s]

95% CI (without gray)
AUROC: (0.712-0.754)
∆AUROC (vs. Ref): (0.024-0.045)
AUPRC: (0.574-0.645)
Brier score: (0.205-0.223)
Brier skill score: (0.023-0.114)
Intercept(calib): (-0.855--0.575)
Slope(calib): (0.690-0.858)
OE ratio: (0.679-0.782)
Spe80% (train):
- Accuracy: (0.680-0.719)
- Sen: (0.560-0.619)
- Spe: (0.737-0.784)
- PPV: (0.547-0.615)
- NPV: (0.737-0.794)
Spe90% (train):
- Accuracy: (0.689-0.732)
- Sen: (0.398-0.458)
- Spe: (0.853-0.886)
- PPV: (0.611-0.686)
- NPV: (0.699-0.757)
Spe95% (train):
- Accuracy: (0.682-0.729)
- Sen: (0.287-0.340)
- Spe: (0.914-0.938)
- PPV: (0.669-0.744)
- NPV: (0.675-0.734)


# ResNet

In [46]:
loaded_model = load_model(model_path / "resnet.model.keras")

In [47]:
prob_y = loaded_model.predict(test_X_s).reshape(-1)

807/807 ━━━━━━━━━━━━━━━━━━━━ 13s 16ms/step


In [48]:
auroc = roc_auc_score(test_y, prob_y)
auprc = average_precision_score(test_y, prob_y)
bss, bs, bs_null = brier_skill_score(test_y, prob_y)
intercept, slope, oe = calib_metrics(test_y, prob_y)
print(f"AUROC: {auroc:.3f}")
print(f"AUPRC: {auprc:.3f}")
print(f"Brier={bs:.3f}, Null Brier={bs_null:.3f}, BSS={bss:.3f}")
print(f"Intercept={intercept:.3f}, Slope={slope:.3f}, OE ratio={oe:.3f}")

AUROC: 0.737
AUPRC: 0.613
Brier=0.335, Null Brier=0.231, BSS=-0.454
Intercept=-2.031, Slope=0.755, OE ratio=0.496


In [49]:
print("Specificity(train): 0.8")
pred = (prob_y >= 0.74).astype(int)  # Spe (train): 80%
tn, fp, fn, tp = confusion_matrix(test_y, pred).ravel()
acc = (tp + tn) / (tp + tn + fp + fn)
sen = tp / (tp + fn)
spe = tn / (tn + fp)
ppv = tp / (tp + fp) if (tp + fp) else np.nan
npv = tn / (tn + fn) if (tn + fn) else np.nan
print(f"Acc: {acc:.3f}")
print(f"Sen: {sen:.3f}")
print(f"Spe: {spe:.3f}")
print(f"PPV: {ppv:.3f}")
print(f"NPV: {npv:.3f}")
print(classification_report(test_y, np.where(prob_y >= 0.74, 1, 0)))

Specificity(train): 0.8
Acc: 0.637
Sen: 0.763
Spe: 0.566
PPV: 0.498
NPV: 0.809
              precision    recall  f1-score   support

         0.0       0.81      0.57      0.67     16498
         1.0       0.50      0.76      0.60      9295

    accuracy                           0.64     25793
   macro avg       0.65      0.66      0.63     25793
weighted avg       0.70      0.64      0.64     25793



In [50]:
print("Specificity(train): 0.9")
pred = (prob_y >= 0.85).astype(int)  # Spe (train): 80%
tn, fp, fn, tp = confusion_matrix(test_y, pred).ravel()
acc = (tp + tn) / (tp + tn + fp + fn)
sen = tp / (tp + fn)
spe = tn / (tn + fp)
ppv = tp / (tp + fp) if (tp + fp) else np.nan
npv = tn / (tn + fn) if (tn + fn) else np.nan
print(f"Acc: {acc:.3f}")
print(f"Sen: {sen:.3f}")
print(f"Spe: {spe:.3f}")
print(f"PPV: {ppv:.3f}")
print(f"NPV: {npv:.3f}")
print(classification_report(test_y, np.where(prob_y >= 0.85, 1, 0)))

Specificity(train): 0.9
Acc: 0.696
Sen: 0.561
Spe: 0.772
PPV: 0.581
NPV: 0.758
              precision    recall  f1-score   support

         0.0       0.76      0.77      0.76     16498
         1.0       0.58      0.56      0.57      9295

    accuracy                           0.70     25793
   macro avg       0.67      0.67      0.67     25793
weighted avg       0.69      0.70      0.70     25793



In [51]:
print("Specificity(train): 0.95")
pred = (prob_y >= 0.91).astype(int)  # Spe (train): 80%
tn, fp, fn, tp = confusion_matrix(test_y, pred).ravel()
acc = (tp + tn) / (tp + tn + fp + fn)
sen = tp / (tp + fn)
spe = tn / (tn + fp)
ppv = tp / (tp + fp) if (tp + fp) else np.nan
npv = tn / (tn + fn) if (tn + fn) else np.nan
print(f"Acc: {acc:.3f}")
print(f"Sen: {sen:.3f}")
print(f"Spe: {spe:.3f}")
print(f"PPV: {ppv:.3f}")
print(f"NPV: {npv:.3f}")
print(classification_report(test_y, np.where(prob_y >= 0.91, 1, 0)))

Specificity(train): 0.95
Acc: 0.709
Sen: 0.398
Spe: 0.885
PPV: 0.660
NPV: 0.723
              precision    recall  f1-score   support

         0.0       0.72      0.88      0.80     16498
         1.0       0.66      0.40      0.50      9295

    accuracy                           0.71     25793
   macro avg       0.69      0.64      0.65     25793
weighted avg       0.70      0.71      0.69     25793



In [52]:
patient_level_bootstrap(
    cid=test_cid,
    seed=config["seed"],
    n=2000,
    ref_y=-lastmbp_test,
    test_y=test_y,
    prob_y=prob_y,
    thrs=[0.74, 0.85, 0.91],
)

100%|██████████████████████████████████| 2000/2000 [00:55<00:00, 35.75it/s]

95% CI (without gray)
AUROC: (0.715-0.759)
∆AUROC (vs. Ref): (0.022-0.057)
AUPRC: (0.579-0.649)
Brier score: (0.320-0.351)
Brier skill score: (-0.574--0.354)
Intercept(calib): (-2.173--1.890)
Slope(calib): (0.678-0.839)
OE ratio: (0.458-0.534)
Spe80% (train):
- Accuracy: (0.615-0.659)
- Sen: (0.736-0.790)
- Spe: (0.531-0.600)
- PPV: (0.464-0.531)
- NPV: (0.779-0.838)
Spe90% (train):
- Accuracy: (0.676-0.717)
- Sen: (0.531-0.594)
- Spe: (0.746-0.797)
- PPV: (0.546-0.617)
- NPV: (0.727-0.786)
Spe95% (train):
- Accuracy: (0.687-0.731)
- Sen: (0.368-0.428)
- Spe: (0.868-0.899)
- PPV: (0.623-0.695)
- NPV: (0.692-0.751)


# LSTMFCN

In [73]:
loaded_model = load_model(model_path / "lstmfcn.model.keras")

In [74]:
prob_y = loaded_model.predict(test_X_s).reshape(-1)

807/807 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step


In [75]:
auroc = roc_auc_score(test_y, prob_y)
auprc = average_precision_score(test_y, prob_y)
bss, bs, bs_null = brier_skill_score(test_y, prob_y)
intercept, slope, oe = calib_metrics(test_y, prob_y)
print(f"AUROC: {auroc:.3f}")
print(f"AUPRC: {auprc:.3f}")
print(f"Brier={bs:.3f}, Null Brier={bs_null:.3f}, BSS={bss:.3f}")
print(f"Intercept={intercept:.3f}, Slope={slope:.3f}, OE ratio={oe:.3f}")

AUROC: 0.721
AUPRC: 0.611
Brier=0.280, Null Brier=0.231, BSS=-0.215
Intercept=-1.442, Slope=0.785, OE ratio=0.564


In [56]:
print("Specificity(train): 0.8")
pred = (prob_y >= 0.58).astype(int)  # Spe (train): 80%
tn, fp, fn, tp = confusion_matrix(test_y, pred).ravel()
acc = (tp + tn) / (tp + tn + fp + fn)
sen = tp / (tp + fn)
spe = tn / (tn + fp)
ppv = tp / (tp + fp) if (tp + fp) else np.nan
npv = tn / (tn + fn) if (tn + fn) else np.nan
print(f"Acc: {acc:.3f}")
print(f"Sen: {sen:.3f}")
print(f"Spe: {spe:.3f}")
print(f"PPV: {ppv:.3f}")
print(f"NPV: {npv:.3f}")
print(classification_report(test_y, np.where(prob_y >= 0.58, 1, 0)))

Specificity(train): 0.8
Acc: 0.594
Sen: 0.792
Spe: 0.483
PPV: 0.463
NPV: 0.804
              precision    recall  f1-score   support

         0.0       0.80      0.48      0.60     16498
         1.0       0.46      0.79      0.58      9295

    accuracy                           0.59     25793
   macro avg       0.63      0.64      0.59     25793
weighted avg       0.68      0.59      0.60     25793



In [57]:
print("Specificity(train): 0.9")
pred = (prob_y >= 0.72).astype(int)  # Spe (train): 80%
tn, fp, fn, tp = confusion_matrix(test_y, pred).ravel()
acc = (tp + tn) / (tp + tn + fp + fn)
sen = tp / (tp + fn)
spe = tn / (tn + fp)
ppv = tp / (tp + fp) if (tp + fp) else np.nan
npv = tn / (tn + fn) if (tn + fn) else np.nan
print(f"Acc: {acc:.3f}")
print(f"Sen: {sen:.3f}")
print(f"Spe: {spe:.3f}")
print(f"PPV: {ppv:.3f}")
print(f"NPV: {npv:.3f}")
print(classification_report(test_y, np.where(prob_y >= 0.72, 1, 0)))

Specificity(train): 0.9
Acc: 0.678
Sen: 0.612
Spe: 0.715
PPV: 0.548
NPV: 0.766
              precision    recall  f1-score   support

         0.0       0.77      0.72      0.74     16498
         1.0       0.55      0.61      0.58      9295

    accuracy                           0.68     25793
   macro avg       0.66      0.66      0.66     25793
weighted avg       0.69      0.68      0.68     25793



In [58]:
print("Specificity(train): 0.95")
pred = (prob_y >= 0.80).astype(int)  # Spe (train): 80%
tn, fp, fn, tp = confusion_matrix(test_y, pred).ravel()
acc = (tp + tn) / (tp + tn + fp + fn)
sen = tp / (tp + fn)
spe = tn / (tn + fp)
ppv = tp / (tp + fp) if (tp + fp) else np.nan
npv = tn / (tn + fn) if (tn + fn) else np.nan
print(f"Acc: {acc:.3f}")
print(f"Sen: {sen:.3f}")
print(f"Spe: {spe:.3f}")
print(f"PPV: {ppv:.3f}")
print(f"NPV: {npv:.3f}")
print(classification_report(test_y, np.where(prob_y >= 0.80, 1, 0)))

Specificity(train): 0.95
Acc: 0.702
Sen: 0.475
Spe: 0.831
PPV: 0.612
NPV: 0.737
              precision    recall  f1-score   support

         0.0       0.74      0.83      0.78     16498
         1.0       0.61      0.47      0.53      9295

    accuracy                           0.70     25793
   macro avg       0.67      0.65      0.66     25793
weighted avg       0.69      0.70      0.69     25793



In [59]:
patient_level_bootstrap(
    cid=test_cid,
    seed=config["seed"],
    n=2000,
    ref_y=-lastmbp_test,
    test_y=test_y,
    prob_y=prob_y,
    thrs=[0.58, 0.72, 0.80],
)

100%|██████████████████████████████████| 2000/2000 [00:56<00:00, 35.65it/s]

95% CI (without gray)
AUROC: (0.697-0.745)
∆AUROC (vs. Ref): (0.008-0.040)
AUPRC: (0.577-0.644)
Brier score: (0.268-0.292)
Brier skill score: (-0.305--0.140)
Intercept(calib): (-1.583--1.301)
Slope(calib): (0.700-0.875)
OE ratio: (0.522-0.608)
Spe80% (train):
- Accuracy: (0.572-0.617)
- Sen: (0.763-0.820)
- Spe: (0.450-0.518)
- PPV: (0.429-0.496)
- NPV: (0.771-0.834)
Spe90% (train):
- Accuracy: (0.658-0.698)
- Sen: (0.577-0.644)
- Spe: (0.688-0.741)
- PPV: (0.512-0.582)
- NPV: (0.734-0.795)
Spe95% (train):
- Accuracy: (0.682-0.723)
- Sen: (0.440-0.509)
- Spe: (0.809-0.850)
- PPV: (0.575-0.647)
- NPV: (0.707-0.766)


# InceptionTime

In [60]:
loaded_model = load_model(model_path / "inceptiontime.model.keras")

In [61]:
prob_y = loaded_model.predict(test_X_s).reshape(-1)

807/807 ━━━━━━━━━━━━━━━━━━━━ 13s 16ms/step


In [62]:
auroc = roc_auc_score(test_y, prob_y)
auprc = average_precision_score(test_y, prob_y)
bss, bs, bs_null = brier_skill_score(test_y, prob_y)
intercept, slope, oe = calib_metrics(test_y, prob_y)
print(f"AUROC: {auroc:.3f}")
print(f"AUPRC: {auprc:.3f}")
print(f"Brier={bs:.3f}, Null Brier={bs_null:.3f}, BSS={bss:.3f}")
print(f"Intercept={intercept:.3f}, Slope={slope:.3f}, OE ratio={oe:.3f}")

AUROC: 0.726
AUPRC: 0.582
Brier=0.289, Null Brier=0.231, BSS=-0.254
Intercept=-1.646, Slope=0.630, OE ratio=0.555


In [63]:
print("Specificity(train): 0.8")
pred = (prob_y >= 0.65).astype(int)  # Spe (train): 80%
tn, fp, fn, tp = confusion_matrix(test_y, pred).ravel()
acc = (tp + tn) / (tp + tn + fp + fn)
sen = tp / (tp + fn)
spe = tn / (tn + fp)
ppv = tp / (tp + fp) if (tp + fp) else np.nan
npv = tn / (tn + fn) if (tn + fn) else np.nan
print(f"Acc: {acc:.3f}")
print(f"Sen: {sen:.3f}")
print(f"Spe: {spe:.3f}")
print(f"PPV: {ppv:.3f}")
print(f"NPV: {npv:.3f}")
print(classification_report(test_y, np.where(prob_y >= 0.65, 1, 0)))

Specificity(train): 0.8
Acc: 0.633
Sen: 0.757
Spe: 0.563
PPV: 0.494
NPV: 0.805
              precision    recall  f1-score   support

         0.0       0.80      0.56      0.66     16498
         1.0       0.49      0.76      0.60      9295

    accuracy                           0.63     25793
   macro avg       0.65      0.66      0.63     25793
weighted avg       0.69      0.63      0.64     25793



In [64]:
print("Specificity(train): 0.9")
pred = (prob_y >= 0.79).astype(int)  # Spe (train): 80%
tn, fp, fn, tp = confusion_matrix(test_y, pred).ravel()
acc = (tp + tn) / (tp + tn + fp + fn)
sen = tp / (tp + fn)
spe = tn / (tn + fp)
ppv = tp / (tp + fp) if (tp + fp) else np.nan
npv = tn / (tn + fn) if (tn + fn) else np.nan
print(f"Acc: {acc:.3f}")
print(f"Sen: {sen:.3f}")
print(f"Spe: {spe:.3f}")
print(f"PPV: {ppv:.3f}")
print(f"NPV: {npv:.3f}")
print(classification_report(test_y, np.where(prob_y >= 0.79, 1, 0)))

Specificity(train): 0.9
Acc: 0.683
Sen: 0.578
Spe: 0.743
PPV: 0.559
NPV: 0.757
              precision    recall  f1-score   support

         0.0       0.76      0.74      0.75     16498
         1.0       0.56      0.58      0.57      9295

    accuracy                           0.68     25793
   macro avg       0.66      0.66      0.66     25793
weighted avg       0.69      0.68      0.68     25793



In [65]:
print("Specificity(train): 0.95")
pred = (prob_y >= 0.86).astype(int)  # Spe (train): 80%
tn, fp, fn, tp = confusion_matrix(test_y, pred).ravel()
acc = (tp + tn) / (tp + tn + fp + fn)
sen = tp / (tp + fn)
spe = tn / (tn + fp)
ppv = tp / (tp + fp) if (tp + fp) else np.nan
npv = tn / (tn + fn) if (tn + fn) else np.nan
print(f"Acc: {acc:.3f}")
print(f"Sen: {sen:.3f}")
print(f"Spe: {spe:.3f}")
print(f"PPV: {ppv:.3f}")
print(f"NPV: {npv:.3f}")
print(classification_report(test_y, np.where(prob_y >= 0.86, 1, 0)))

Specificity(train): 0.95
Acc: 0.694
Sen: 0.434
Spe: 0.841
PPV: 0.606
NPV: 0.725
              precision    recall  f1-score   support

         0.0       0.73      0.84      0.78     16498
         1.0       0.61      0.43      0.51      9295

    accuracy                           0.69     25793
   macro avg       0.67      0.64      0.64     25793
weighted avg       0.68      0.69      0.68     25793



In [66]:
patient_level_bootstrap(
    cid=test_cid,
    seed=config["seed"],
    n=2000,
    ref_y=-lastmbp_test,
    test_y=test_y,
    prob_y=prob_y,
    thrs=[0.65, 0.79, 0.86],
)

100%|██████████████████████████████████| 2000/2000 [00:55<00:00, 36.10it/s]

95% CI (without gray)
AUROC: (0.700-0.752)
∆AUROC (vs. Ref): (0.004-0.052)
AUPRC: (0.540-0.624)
Brier score: (0.273-0.304)
Brier skill score: (-0.357--0.169)
Intercept(calib): (-1.806--1.494)
Slope(calib): (0.550-0.720)
OE ratio: (0.513-0.596)
Spe80% (train):
- Accuracy: (0.610-0.656)
- Sen: (0.726-0.787)
- Spe: (0.527-0.601)
- PPV: (0.459-0.527)
- NPV: (0.774-0.833)
Spe90% (train):
- Accuracy: (0.661-0.705)
- Sen: (0.539-0.615)
- Spe: (0.711-0.772)
- PPV: (0.516-0.598)
- NPV: (0.726-0.786)
Spe95% (train):
- Accuracy: (0.672-0.717)
- Sen: (0.399-0.471)
- Spe: (0.816-0.863)
- PPV: (0.561-0.650)
- NPV: (0.695-0.754)


# XGB vs. ResNet

In [33]:
loaded_xgb = XGBClassifier()
loaded_xgb.load_model(model_path / "xgb_baseline.json")
loaded_model = load_model(model_path / "resnet.model.keras")

2026-10-09 09:35:15.236652: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M5 Max
2026-10-09 09:35:15.236681: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 128.00 GB
2026-10-09 09:35:15.236691: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 53.76 GB
2026-10-09 09:35:15.236712: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-10-09 09:35:15.236720: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [34]:
xgb_prob_y = loaded_xgb.predict_proba(test_X_feat)[:, 1]
resnet_prob_y = loaded_model.predict(test_X_s).reshape(-1)

2026-10-09 09:35:17.411452: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


807/807 ━━━━━━━━━━━━━━━━━━━━ 27s 33ms/step


In [35]:
patient_level_bootstrap(
    cid=test_cid,
    seed=config["seed"],
    n=2000,
    ref_y=resnet_prob_y,
    test_y=test_y,
    prob_y=xgb_prob_y,
    thrs=[0.59, 0.72, 0.82],
)

100%|██████████████████████████████████| 2000/2000 [01:15<00:00, 26.60it/s]

95% CI (without gray)
AUROC: (0.721-0.761)
∆AUROC (vs. Ref): (-0.007-0.014)
AUPRC: (0.585-0.648)
Brier score: (0.214-0.232)
Brier skill score: (-0.023-0.079)
Intercept(calib): (-1.055--0.764)
Slope(calib): (0.626-0.757)
OE ratio: (0.643-0.740)
Spe80% (train):
- Accuracy: (0.666-0.701)
- Sen: (0.606-0.663)
- Spe: (0.687-0.735)
- PPV: (0.519-0.584)
- NPV: (0.745-0.802)
Spe90% (train):
- Accuracy: (0.688-0.728)
- Sen: (0.442-0.503)
- Spe: (0.824-0.857)
- PPV: (0.592-0.658)
- NPV: (0.709-0.766)
Spe95% (train):
- Accuracy: (0.685-0.730)
- Sen: (0.301-0.356)
- Spe: (0.910-0.931)
- PPV: (0.667-0.734)
- NPV: (0.679-0.737)


# LSTMFCN vs. InceptionTime

In [70]:
loaded_lstmfcn = load_model(model_path / "lstmfcn.model.keras")
loaded_inceptiontime = load_model(model_path / "inceptiontime.model.keras")

In [71]:
lstmfcn_prob_y = loaded_lstmfcn.predict(test_X_s).reshape(-1)
inceptiontime_prob_y = loaded_inceptiontime.predict(test_X_s).reshape(-1)

807/807 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step
807/807 ━━━━━━━━━━━━━━━━━━━━ 13s 16ms/step


In [72]:
patient_level_bootstrap(
    cid=test_cid,
    seed=config["seed"],
    n=2000,
    ref_y=inceptiontime_prob_y,
    test_y=test_y,
    prob_y=lstmfcn_prob_y,
    thrs=[0.58, 0.72, 0.80],
)

100%|██████████████████████████████████| 2000/2000 [00:55<00:00, 36.05it/s]

95% CI (without gray)
AUROC: (0.697-0.745)
∆AUROC (vs. Ref): (-0.023-0.014)
AUPRC: (0.577-0.644)
Brier score: (0.268-0.292)
Brier skill score: (-0.305--0.140)
Intercept(calib): (-1.583--1.301)
Slope(calib): (0.700-0.875)
OE ratio: (0.522-0.608)
Spe80% (train):
- Accuracy: (0.572-0.617)
- Sen: (0.763-0.820)
- Spe: (0.450-0.518)
- PPV: (0.429-0.496)
- NPV: (0.771-0.834)
Spe90% (train):
- Accuracy: (0.658-0.698)
- Sen: (0.577-0.644)
- Spe: (0.688-0.741)
- PPV: (0.512-0.582)
- NPV: (0.734-0.795)
Spe95% (train):
- Accuracy: (0.682-0.723)
- Sen: (0.440-0.509)
- Spe: (0.809-0.850)
- PPV: (0.575-0.647)
- NPV: (0.707-0.766)


# XGB vs. CNN

In [36]:
loaded_xgb = XGBClassifier()
loaded_xgb.load_model(model_path / "xgb_baseline.json")
loaded_model = load_model(model_path / "cnn.model.keras")

In [37]:
xgb_prob_y = loaded_xgb.predict_proba(test_X_feat)[:, 1]
cnn_prob_y = loaded_model.predict(test_X_s).reshape(-1)

807/807 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step


In [38]:
patient_level_bootstrap(
    cid=test_cid,
    seed=config["seed"],
    n=2000,
    ref_y=cnn_prob_y,
    test_y=test_y,
    prob_y=xgb_prob_y,
    thrs=[0.59, 0.72, 0.82],
)

100%|██████████████████████████████████| 2000/2000 [01:18<00:00, 25.61it/s]

95% CI (without gray)
AUROC: (0.721-0.761)
∆AUROC (vs. Ref): (-0.001-0.018)
AUPRC: (0.585-0.648)
Brier score: (0.214-0.232)
Brier skill score: (-0.023-0.079)
Intercept(calib): (-1.055--0.764)
Slope(calib): (0.626-0.757)
OE ratio: (0.643-0.740)
Spe80% (train):
- Accuracy: (0.666-0.701)
- Sen: (0.606-0.663)
- Spe: (0.687-0.735)
- PPV: (0.519-0.584)
- NPV: (0.745-0.802)
Spe90% (train):
- Accuracy: (0.688-0.728)
- Sen: (0.442-0.503)
- Spe: (0.824-0.857)
- PPV: (0.592-0.658)
- NPV: (0.709-0.766)
Spe95% (train):
- Accuracy: (0.685-0.730)
- Sen: (0.301-0.356)
- Spe: (0.910-0.931)
- PPV: (0.667-0.734)
- NPV: (0.679-0.737)


In [39]:
# Patient-level bootstrap resampling
pat_ids = np.unique(test_cid)
idx_by_pat = {p: np.where(test_cid == p)[0] for p in pat_ids}
np.random.seed(config["seed"])  # 42

auroc_list, delta_list, auprc_list = [], [], []
bs_list, bss_list = [], []
icpt_list, slope_list, oe_list = [], [], []
acc_80, sen_80, spe_80, ppv_80, npv_80 = [], [], [], [], []
acc_90, sen_90, spe_90, ppv_90, npv_90 = [], [], [], [], []
acc_95, sen_95, spe_95, ppv_95, npv_95 = [], [], [], [], []
for i in tqdm(range(n), ncols=75):
    sampled = np.random.choice(pat_ids, size=len(pat_ids), replace=True)
    idx = np.concatenate([idx_by_pat[p] for p in sampled])
    sampled_ref_test = cnn_prob_y[idx]
    sampled_test_y, sampled_prob_y = test_y[idx], xgb_prob_y[idx]

    ref_bss, ref_bs, ref_bs_null = brier_skill_score(sampled_test_y, sampled_ref_test)
    bss, bs, bs_null = brier_skill_score(sampled_test_y, sampled_prob_y)

    bss_list.append(bss)
    delta_list.append(bss - ref_bss)

print("95% CI (without gray)")
print(
    f"Brier skill score: ({np.percentile(delta_list, 2.5):.3f}-{np.percentile(delta_list, 97.5):.3f})"
)

100%|█████████████████████████████████| 2000/2000 [00:02<00:00, 788.49it/s]

95% CI (without gray)
Brier skill score: (-0.057--0.021)
